# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 614)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
#                                                                                                         for file_name in file_list)), 
#                        columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()
full_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names])

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,S_0,I_0,b_I,d_S,d_I,d_IE,K_I,d_H,K_H,N_0,max_Na,t_bind,t_unbind,t_Na_div,t_E_div,t_M_div,t_E_die,t_cycle,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,p_load,T_max_pI,T_min_pI,harm_pI,harm_pS,max_pE,T_pE_max,T_pE_start,max_eM,T_pEcyteM,T_pE_end,frac_cM,int_pHE,int_pHI,E_end
0,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.2,-0.8,0.4,-0.6,-1.2,-0.8,0.4,-1.2,-0.0,-0.0,0.0,-27.0,1.2,0.8,-0.4,-1.8,-5.684342e-17,0.00,0.02,1.020000e+03,2.069314e+06,165243.0,17.66,2.07,0.0,0.0,25.97,0.681818,154520.999874,256.786870,1.0
1,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.2,-0.8,0.4,-0.6,-1.2,-0.8,0.4,-1.2,-0.0,-0.0,0.0,-27.0,1.2,0.8,-0.4,-1.2,5.684342e-17,0.00,0.02,1.020000e+03,1.784812e+06,230698.0,22.28,0.50,0.0,0.0,28.41,0.602740,159312.495906,256.774379,1.0
2,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.2,-0.8,0.4,-0.6,-1.2,-0.8,0.4,-1.2,-0.0,-0.0,0.0,-27.0,1.2,0.8,-0.4,-0.6,1.762146e-14,0.00,0.02,1.020000e+03,7.570706e+03,1959.0,10.60,0.62,0.0,0.0,0.00,0.663793,217.792833,256.785647,1.0
3,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.2,-0.8,0.4,-0.6,-1.2,-0.8,0.4,-1.2,-0.0,-0.0,0.0,-27.0,1.2,0.8,-0.4,0.0,7.997869e-14,0.00,0.02,1.020000e+03,1.405194e+02,234.0,2.66,0.57,0.0,0.0,0.00,0.740741,16.900800,256.786163,1.0
4,10000000.0,1000.0,1.000000e-07,0.01,1.0,12.0,10000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,-1.2,-0.8,0.4,-0.6,-1.2,-0.8,0.4,-1.2,-0.0,-0.0,0.0,-27.0,1.2,0.8,-0.4,0.6,8.077450e-14,0.00,0.02,1.020000e+03,8.531973e+01,224.0,8.02,0.40,0.0,0.0,0.00,0.793478,10.163615,256.786870,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177156095,10000000.0,1000.0,1.000000e-07,0.01,0.2,12.0,10000000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.4,-1.6,0.8,0.6,0.4,-1.6,0.8,0.6,0.0,-0.0,0.0,-27.0,-0.4,1.6,-0.8,-1.2,5.484528e+02,13.63,0.00,1.095591e+07,4.326578e+04,12547.0,6.02,0.55,0.0,0.0,7.25,0.641975,7756.623619,480845.505865,1.0
177156096,10000000.0,1000.0,1.000000e-07,0.01,0.2,12.0,10000000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.4,-1.6,0.8,0.6,0.4,-1.6,0.8,0.6,0.0,-0.0,0.0,-27.0,-0.4,1.6,-0.8,-0.6,5.474039e+02,13.60,0.00,1.098459e+07,1.534916e+04,5214.0,5.19,0.48,0.0,0.0,0.00,0.638498,2320.923494,482987.481155,1.0
177156097,10000000.0,1000.0,1.000000e-07,0.01,0.2,12.0,10000000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.4,-1.6,0.8,0.6,0.4,-1.6,0.8,0.6,0.0,-0.0,0.0,-27.0,-0.4,1.6,-0.8,0.0,5.467275e+02,13.59,0.00,1.099918e+07,1.050680e+03,582.0,2.56,0.92,0.0,0.0,0.00,0.686695,130.100839,484053.752375,1.0
177156098,10000000.0,1000.0,1.000000e-07,0.01,0.2,12.0,10000000.0,2.0,10000.0,100.0,4.0,0.5,1.0,0.366667,0.333333,0.5,10.0,0.25,0.4,-1.6,0.8,0.6,0.4,-1.6,0.8,0.6,0.0,-0.0,0.0,-27.0,-0.4,1.6,-0.8,0.6,5.466965e+02,13.59,0.00,1.099973e+07,5.059217e+02,378.0,4.36,0.88,0.0,0.0,0.00,0.668103,58.796630,484093.428744,1.0


In [8]:
# Create additional variables
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

full_df['antigenicity_over_harm'] = antigenicity_over_harm(full_df)
full_df['stim_pI'] = np.log(1 + (full_df['p_load']/full_df['K_I']))
full_df['stim_pHI'] = np.log(1 + (full_df['int_pHI']/full_df['K_H']))
full_df['stim_pHE'] = np.log(1 + (full_df['int_pHE']/full_df['K_H']))
#full_df['scaled_min_pS'] = full_df['min_pS']/full_df['S_0']

# identify Biologically evidenced networks
keep_vars = ['harm_pI', 'harm_pS', 'frac_cM', 'max_pE',
             'T_pE_start', 'T_pE_max', 'T_pE_end',
             'stim_pI', 'stim_pHI', 'stim_pHE',
             'E_end', 'antigenicity_over_harm']

In [9]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_0

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + keep_vars]

    # compute infection harm without T cell response
    no_eff_data[l] = lin_stoch_sim(N_0 = 0, I_0 = I_0, K_I = K_I, d_I = d_I, b_I = b_I,
                                   infection_model = "cancer" if b_I >= b_C else "acute")
    no_eff_stats = no_eff_data[l]["summary_stats"]

    data.loc[:,"harm_pI_noprotection"] = no_eff_stats[3]/S_0
    data.loc[:,"peff_clearance"] = (no_eff_stats[3] - data['harm_pI'].to_numpy())/S_0
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_0
    data.loc[:,"peff_protection"] = data['peff_clearance'] - data['peff_toxicity']
    data.loc[:,"peff_scaled_protection"] = data['peff_protection']/data['harm_pI_noprotection']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [31:03<00:00, 18.63s/it]


In [10]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios